# Import the libraries and the dataset

In [1]:
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import LinearSVR
import numpy as np
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.utils import shuffle
import math

In [2]:
df=pd.read_csv('truedata.csv')

In [3]:
df = df.dropna()

In [4]:
df

,Strain Rate,Temperature,True Strain,True Stress,True Plastic Strain
0,0.0001,27.0,0.10683,939.31764,0.00030
1,0.0001,27.0,0.10683,939.32333,0.00030
2,0.0001,27.0,0.10684,939.33680,0.00031
3,0.0001,27.0,0.10684,939.34085,0.00031
4,0.0001,27.0,0.10685,939.34472,0.00032
...,...,...,...,...,...
162754,0.0100,500.0,0.06196,597.07372,0.01804
162755,0.0100,500.0,0.06218,597.09635,0.01826
162756,0.0100,500.0,0.06239,597.11009,0.01847
162757,0.0100,500.0,0.06260,597.11561,0.01868


In [5]:
TargetVariable=['True Stress']
Predictors=['Strain Rate','Temperature','True Plastic Strain']
 
X=df[Predictors].values
y=df[TargetVariable].values

# Hyperparameter tuning using Grid Search and 10 fold cross validation

In [6]:
from sklearn.preprocessing import StandardScaler
PredictorScaler=StandardScaler()
TargetVarScaler=StandardScaler()
 
# Storing the fit object for later reference
PredictorScalerFit=PredictorScaler.fit(X)
TargetVarScalerFit=TargetVarScaler.fit(y)
 
# Generating the standardized values of X and y
X=PredictorScalerFit.transform(X)
y=TargetVarScalerFit.transform(y)

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test  = train_test_split(X, y, test_size=0.3, random_state=42)

In [9]:
def svr_model(X_train, y_train):
    gsc = GridSearchCV(LinearSVR(),
        param_grid={
            'C': [ 0.01, 0.1, 1, 10, 100, 500],
            'epsilon': [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10],
        },
        cv=10, scoring='neg_mean_squared_error', verbose=0, n_jobs=-1)

    grid_result = gsc.fit(X_train, y_train)
    best_params = grid_result.best_params_
    print(grid_result.best_params_)
    best_svr = LinearSVR(C=best_params["C"], epsilon=best_params["epsilon"], 
                         tol=0.001, max_iter=-1)          

    scoring = {
               'abs_error': 'neg_mean_absolute_error',
               'squared_error': 'neg_mean_squared_error'}

    scores = cross_validate(best_svr, X_train, y_train, cv=10, scoring=scoring, return_train_score=True)  
    return "MAE :", abs(scores['test_abs_error'].mean()), "| RMSE :", math.sqrt(abs(scores['test_squared_error'].mean()))


    

In [10]:
print(svr_model(X_train,y_train))

{'C': 0.01, 'epsilon': 0.5}
('MAE :', 0.82424469829574, '| RMSE :', 1.0002307350864148)


C:\Users\ankit\Downloads\Anaconda\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\ankit\Downloads\Anaconda\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\ankit\Downloads\Anaconda\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\ankit\Downloads\Anaconda\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=Tru

# Refitting the model with the best parameters and evaluating the model

In [8]:
from sklearn.svm import LinearSVR
regressor = LinearSVR(C=0.01, epsilon=0.5)
regressor.fit(X_train, y_train)

C:\Users\ankit\Downloads\Anaconda\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


LinearSVR(C=0.01, epsilon=0.5)

In [9]:
y_pred = regressor.predict(X_test)

In [10]:
y_pred

array([-0.84381442, -0.65914171, -0.51092321, ...,  0.15233368,
        2.06882615, -0.70390208])

In [11]:
y_pred = y_pred.reshape(-1,1)

In [12]:
y_pred = TargetVarScalerFit.inverse_transform(y_pred)

In [13]:
y2_test = TargetVarScalerFit.inverse_transform(y_test)

In [14]:
y2_test

array([[ 781.83656],
       [ 622.26907],
       [ 694.36165],
       ...,
       [ 740.82561],
       [1152.24326],
       [ 950.02404]])

In [15]:
APE=100*(abs(y2_test-y_pred)/y2_test)

In [16]:
print('The Accuracy of SVR model is:', np.mean(APE))

The Accuracy of SVR model is: 11.431716277530652


In [17]:
MAE=(abs(y2_test-y_pred))
print('The Accuracy of SVR model is:',np.mean(MAE))

The Accuracy of SVR model is: 94.27689827839814


In [18]:
y2_test

array([[ 781.83656],
       [ 622.26907],
       [ 694.36165],
       ...,
       [ 740.82561],
       [1152.24326],
       [ 950.02404]])

In [22]:
y_pred = y_pred.reshape(-1)

In [23]:
y_pred

array([ 684.7066882 ,  712.75954255,  735.27479267, ...,  836.02736195,
       1127.15368308,  705.96018395])

In [24]:
y2_test = y2_test.reshape(-1)

In [25]:
y2_test

array([ 781.83656,  622.26907,  694.36165, ...,  740.82561, 1152.24326,
        950.02404])

In [27]:
import statistics
var = (statistics.variance(y_pred))
print(var)
chi_sq = np.sum(((y2_test-y_pred)**2)/var)
red_chi_sq = chi_sq/5074
print('The reduced chi squared value for SVR is', red_chi_sq)

12018.753709172497
The reduced chi squared value for SVR is 10.07966113159387


# Validating the model on new data

In [74]:
dfvalid=pd.read_csv('150Cvalidationdata.csv')

In [75]:
Predictors=['Strain Rate','Temperature','True Plastic Strain']
X_valid=dfvalid[Predictors].values

In [76]:
X_valid

array([[1.0000000e-03, 1.5000000e+02, 6.1900000e-05],
       [1.0000000e-03, 1.5000000e+02, 5.2700000e-05],
       [1.0000000e-03, 1.5000000e+02, 6.9800000e-05],
       ...,
       [1.0000000e-03, 1.5000000e+02, 9.4723457e-02],
       [1.0000000e-03, 1.5000000e+02, 9.4695481e-02],
       [1.0000000e-03, 1.5000000e+02, 9.4743924e-02]])

In [77]:
from sklearn.preprocessing import StandardScaler
PredictorScaler=StandardScaler()
TargetVarScaler=StandardScaler()
 
# Storing the fit object for later reference
PredictorScalerFit=PredictorScaler.fit(X_valid)

# Generating the standardized values of X and y
X_valid=PredictorScalerFit.transform(X_valid)

In [78]:
X_valid

array([[-2.16840434e-19,  0.00000000e+00, -1.75357897e+00],
       [-2.16840434e-19,  0.00000000e+00, -1.75391575e+00],
       [-2.16840434e-19,  0.00000000e+00, -1.75328979e+00],
       ...,
       [-2.16840434e-19,  0.00000000e+00,  1.71155471e+00],
       [-2.16840434e-19,  0.00000000e+00,  1.71053064e+00],
       [-2.16840434e-19,  0.00000000e+00,  1.71230392e+00]])

In [79]:
y_pred_valid = regressor.predict(X_valid)

In [80]:
y_pred_valid

array([-1.28860501, -1.28884099, -1.28840237, ...,  1.13946663,
        1.13874904,  1.13999161])

In [81]:
y_pred_valid = y_pred_valid.reshape(-1,1)

In [82]:
y_pred_valid = TargetVarScalerFit.inverse_transform(y_pred_valid)

In [83]:
y_pred_valid

array([[617.14041722],
       [617.10457045],
       [617.17119868],
       ...,
       [985.97858103],
       [985.86957568],
       [986.05832841]])

In [84]:
import pandas as pd 
pd.DataFrame(y_pred_valid).to_csv("svrvalidationpredicteddata.csv")